In [1]:
pip install flask-ngrok

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
cd /content/drive/MyDrive/Colab Notebooks/project AI

/content/drive/MyDrive/Colab Notebooks/project AI


In [5]:
from flask_ngrok import run_with_ngrok
from flask import Flask, request, render_template
import pickle
import numpy as np

In [6]:
!pip install pyngrok

In [7]:
!ngrok authtoken 33AxSUJJwHVi6hM4V4O6yUR8YQq_6MCJtCfrkZAYk8WwzwS9S

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [8]:
import os, io
from datetime import datetime
from werkzeug.utils import secure_filename


In [9]:
import numpy as np  # ใช้สำหรับคำนวณเชิงตัวเลขและจัดการอาร์เรย์
from flask import Flask, render_template, request  # นำเข้า Flask และฟังก์ชันสำหรับเรนเดอร์หน้า/รับค่า request
from tensorflow.keras.models import load_model  # ฟังก์ชันโหลดโมเดล Keras (แบ็กเอนด์ TensorFlow)
from tensorflow.keras.utils import load_img , img_to_array  # ฟังก์ชันโหลดรูปและแปลงรูปเป็นอาร์เรย์
from keras.models import load_model  # นำเข้า load_model จาก keras อีกครั้ง (ซ้ำกับบรรทัดบน คงไว้ตามต้นฉบับ)
from pyngrok import ngrok   # ใช้สร้าง public URL ผ่าน ngrok เพื่อให้ภายนอกเข้าถึงเซิร์ฟเวอร์โลคอลได้

app = Flask(__name__)  # สร้างแอป Flask

run_with_ngrok(app)  # เรียกใช้เพื่อให้ Flask ทำงานร่วมกับ ngrok (Run with ngrok = รันผ่าน ngrok)

model = load_model('vehicle_classification_model1 (1).keras')  # โหลดโมเดลที่เทรนแล้วจากไฟล์

model.make_predict_function()  # เตรียมฟังก์ชัน predict (make_predict_function = เตรียมฟังก์ชันทำนายล่วงหน้า)
ngrok_tunnel = ngrok.connect(5000)  # เปิดอุโมงค์ ngrok ที่พอร์ต 5000

# Print the public URL  ← แปล: "พิมพ์/แสดง URL แบบสาธารณะ"
print('Public URL:', ngrok_tunnel.public_url)  # แสดงลิงก์สาธารณะของแอป (ไว้กดเปิดดูเว็บจากภายนอก/บนมือถือ/คนอื่น)

def predict_label(img_path):
    # เตรียมรูปภาพให้เข้ากับโมเดล
    test_image = load_img(img_path, target_size=(224, 224))  # โหลดรูปและปรับขนาด
    test_image = img_to_array(test_image) / 255.0            # แปลงเป็นอาร์เรย์ float และ normalize เป็น [0,1]
    test_image = test_image.reshape(1, 224, 224, 3)          # เพิ่มมิติ batch เป็น (1, 224, 224, 3)

    # พยากรณ์ความน่าจะเป็นของทั้ง 3 คลาส
    probs = model.predict(test_image)[0]             # ได้เวกเตอร์ความน่าจะเป็นขนาด 3 เช่น [0.01, 0.9, 0.1]
    predicted_class_index = int(np.argmax(probs))   # หา index ของค่ามากสุด (0/1/2)

    # ค่าความมั่นใจอันดับ 1 และส่วนต่างกับอันดับ 2
    top1 = float(probs[predicted_class_index]) * 100  # ค่าความมั่นใจของคลาสที่โมเดลเลือก
    top2 = float(np.sort(probs)[-2]) * 100            # ค่าความมั่นใจอันดับสอง (เอาค่าเรียงแล้วหยิบตัวรองสุดท้าย)
    margin = top1 - top2                              # ส่วนต่างระหว่างอันดับ 1 และ 2 (บ่งชี้ “ความแน่ใจ” ของโมเดล)

    # ---- เกณฑ์ reject ----
    THRESHOLD = 80.0   # ถ้าความมั่นใจต่ำกว่า 80% ให้ขึ้น "ไม่สามารถจำแนกได้"
    DELTA = 10.0       # ถ้าส่วนต่าง Top1-Top2 ต่ำกว่า 10% ให้ขึ้น "ไม่สามารถจำแนกได้"

    if (top1 < THRESHOLD) or (margin < DELTA):
        prediction = 'ไม่สามารถจำแนกรูปภาพได้'  # แสดงผลแบบ “reject” ให้ผู้ใช้ทราบ
        confidence = top1                    # ยังส่ง top1 กลับเพื่อประกอบการตัดสินใจ/แสดงผ
        print(prediction)
        print(f"Predicted index: {predicted_class_index}, confidence: {confidence:.2f}% (rejected, margin {margin:.2f}%)")
        return confidence, prediction

    # แมปดัชนีเป็นชื่อคลาส
    # หมายเหตุ: ควรให้ดัชนีสอดคล้องกับลำดับที่ใช้ตอนเทรน (เช่น class_indices ของ ImageDataGenerator)
    if predicted_class_index == 0:
        prediction = 'จักรยาน'
    elif predicted_class_index == 1:
        prediction = 'รถยนต์'
    elif predicted_class_index == 2:
        prediction = 'จักรยานยนต์'
    else:
        prediction = 'ไม่สามารถจำแนกรูปภาพได้'   # กันพลาดกรณีดัชนีอยู่นอกช่วง

    confidence = top1 # ส่งความมั่นใจของคลาสที่ทำนายเป็นเปอร์เซ็นต์
    print(prediction)
    print(f"Predicted index: {predicted_class_index}, confidence: {confidence:.2f}% (margin {margin:.2f}%)")
    return confidence, prediction


# ----- ส่วนกำหนดเส้นทาง (Routing) ของเว็บ -----
@app.route("/", methods=['GET', 'POST'])  # หน้าแรก รองรับทั้ง GET และ POST
def main():
  # เรนเดอร์ไฟล์เทมเพลต templates/index.html
  return render_template("index.html")  # เรนเดอร์ไฟล์เทมเพลต index.html (render template)



@app.route("/submit", methods = ['GET', 'POST'])  # เส้นทางรับอัปโหลดรูปจากฟอร์ม (ชื่อฟิลด์ my_image)
def get_output():
  if request.method == 'POST':  # เมื่อส่งฟอร์มแบบ POST
    img = request.files['my_image']   # ดึงไฟล์รูปที่ผู้ใช้อัปโหลด (input name="my_image" ในฟอร์ม)

    img_path = "static/uploads/" + img.filename  # ระบุพาธที่จะบันทึกรูป (ควบรวมโฟลเดอร์กับชื่อไฟล์)

    img.save(img_path)  # สั่งเซฟไฟล์รูปลงปลายทาง (save uploaded file)

    p1 = predict_label(img_path)  # เรียกโมเดลเพื่อทำนาย -> ได้ (confidence, label)


  # render_template(...): ส่งค่ากลับไปแสดงในหน้า index.html → prediction = ค่าความมั่นใจ, TEXT = ป้ายชื่อคลาส, img_path = พาธรูป
  return render_template("index.html", prediction = p1[0], TEXT = p1[1], img_path = img_path)

app.run()  # รันเซิร์ฟเวอร์ Flask (start Flask server)


Public URL: https://hydroelectric-elvira-enthusiastically.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


 * Running on http://hydroelectric-elvira-enthusiastically.ngrok-free.dev
 * Traffic stats available on http://127.0.0.1:4040


INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 08:13:06] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 08:13:07] "GET /static/images/car1.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 08:13:07] "GET /static/images/motor.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 08:13:07] "GET /static/images/car.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 08:13:07] "GET /static/images/bike.png HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Sep/2026 08:13:08] "GET /favicon.ico HTTP/1.1" 404 -
